In [1]:
## 사전코드 - 책에는 없습니다.
import plotly.io as pio
pio.renderers.default = "notebook_connected"


In [ ]:
## P362
import duckdb
import polars as pl
import plotly.express as px

con_youtube = duckdb.connect("./youtube.duckdb") ## youtube.duckdb 파일의 경로명은 각자 설정
daily_rank = con_youtube.sql("""
    FROM youtube
    SELECT daily_rank""").pl()
fig = px.histogram(daily_rank, nbins = 50)
fig.update_traces(
    marker_line_color="white", # 테두리 색
    marker_line_width=1 # 테두리 두께
)
fig.show()

In [ ]:
## P363
fig = px.histogram(con_youtube.sql('''from youtube select daily_movement''').pl(), 
    nbins = 50)
fig.show()

In [ ]:
## P364
fig = px.histogram(con_youtube.sql('''from youtube select weekly_movement''').pl(), 
    nbins = 50)
fig.show()

In [ ]:
## P365 상단
fig = px.histogram(con_youtube.sql('''from youtube select snapshot_date''').pl(), 
    nbins = 100)
fig.show()

In [6]:
## P365 하단
con_youtube.sql("""
    from youtube
    select snapshot_date, count(*)
    where snapshot_date between '2024-06-30' and '2024-07-06'
    group by snapshot_date
    order by snapshot_date
""").pl()

snapshot_date,count_star()
date,i64
2024-06-30,5650
2024-07-01,5650
2024-07-02,5650
2024-07-03,5650
2024-07-04,5650
2024-07-06,5650


In [ ]:
### P366
view_count = con_youtube.sql("""
    FROM youtube
    SELECT view_count""").pl()
fig = px.histogram(view_count, nbins = 100)
fig.show()

In [ ]:
## P367
view_count = con_youtube.sql("""
    FROM youtube
    SELECT view_count, log10(view_count) as log
    WHERE view_count > 0""").pl()
fig = px.histogram(view_count, nbins = 100, x = 'log')
fig.show()

In [ ]:
## P368
comment_daily_change = con_youtube.sql("""
    FROM youtube
    SELECT comment_daily_change, log10(comment_daily_change) as log
    WHERE comment_daily_change > 0""").pl()
fig = px.histogram(comment_daily_change, nbins = 100, x = 'log')
fig.show()

In [ ]:
## P370
view_like_scatter = con_youtube.sql("""
    FROM youtube
    SELECT view_count, like_count
    USING SAMPLE (10%) REPEATABLE (123);""").pl()
fig = px.scatter(view_like_scatter, x = 'view_count', y = 'like_count', opacity=0.1)
fig.show()

In [ ]:
## P371
view_like_scatter = con_youtube.sql("""
    FROM youtube
    SELECT view_count, like_count
    USING SAMPLE (10%) REPEATABLE (123);""").pl()
fig = px.scatter(view_like_scatter, x = 'view_count', y = 'like_count', log_x=True, log_y=True, trendline='ols', opacity=0.005)
fig.show()

In [12]:
## P372
con_youtube.sql('''
    FROM youtube
    SELECT DOW_publish_date, "재생 수 합계":sum(view_count), "좋아요 수 합계":sum(like_count),
        "댓글 수 합계":sum(comment_count)
    GROUP BY DOW_publish_date
    ORDER BY "재생 수 합계" DESC''').pl()

DOW_publish_date,재생 수 합계,좋아요 수 합계,댓글 수 합계
str,"decimal[38,0]","decimal[38,0]","decimal[38,0]"
"""토""",9018784488451,292373229980,5973664143
"""금""",7811383302237,256873679134,9328003726
"""목""",7403019548731,227011578071,3473223857
"""수""",6619227491486,206878053516,2945083304
"""화""",6509447703439,191536138839,3026920299
"""월""",6411256320269,194596839720,4089970858
"""일""",5777353902822,162745717571,2592932790


In [13]:
## P373
con_youtube.sql('''
    FROM youtube
    SELECT DOW_snapshot_date, "재생 수 변동량":sum(view_daily_change),
        "좋아요 수 변동량":sum(like_daily_change), "댓글 수 변동량":sum(comment_daily_change)
    GROUP BY DOW_snapshot_date
    ORDER BY "재생 수 변동량" DESC''').pl()

DOW_snapshot_date,재생 수 변동량,좋아요 수 변동량,댓글 수 변동량
str,"decimal[38,0]","decimal[38,0]","decimal[38,0]"
"""월""",612981181039,16101695350,216469649
"""일""",603259191393,15847423185,225932767
"""화""",558093488765,14419805786,186094166
"""토""",553100282962,14641842751,206227543
"""수""",519589235178,12946658261,159389774
"""목""",503663683884,12457181795,151152733
"""금""",448770212191,12181448520,149966773


In [14]:
## P374
con_youtube.sql("""
    SELECT corr(view_count, like_count) AS pearson_corr
    FROM youtube
    WHERE country = 'KR'""").pl()

pearson_corr
f64
0.932545


In [15]:
## P375
corr_youtube = con_youtube.sql("""
    FROM youtube
    SELECT daily_rank, daily_movement, weekly_movement, view_count, like_count,
        comment_count, comment_rate, like_rate, view_daily_change,
        like_daily_change, comment_daily_change
    WHERE columns(['view_daily_change', 'like_daily_change', 'comment_daily_change']) IS NOT NULL 
AND country = 'KR' """).pl().corr()
corr_youtube

daily_rank,daily_movement,weekly_movement,view_count,like_count,comment_count,comment_rate,like_rate,view_daily_change,like_daily_change,comment_daily_change
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1.0,-0.33866,-0.943702,-0.051866,-0.054413,-0.089073,-0.037878,-0.058277,-0.157199,-0.132183,-0.154446
-0.33866,1.0,0.288848,0.033634,0.041488,0.070271,0.008907,0.081837,0.118136,0.098966,0.106873
-0.943702,0.288848,1.0,0.000093,0.010324,0.056401,0.04013,0.059522,0.129613,0.112871,0.13633
-0.051866,0.033634,0.000093,1.0,0.936132,0.666573,-0.012031,0.07703,0.74706,0.597088,0.398143
-0.054413,0.041488,0.010324,0.936132,1.0,0.784505,0.031759,0.18842,0.692405,0.690308,0.494301
…,…,…,…,…,…,…,…,…,…,…
-0.037878,0.008907,0.04013,-0.012031,0.031759,0.367674,1.0,0.391542,-0.000397,0.041766,0.229977
-0.058277,0.081837,0.059522,0.07703,0.18842,0.21685,0.391542,1.0,0.090533,0.207485,0.190228
-0.157199,0.118136,0.129613,0.74706,0.692405,0.557755,-0.000397,0.090533,1.0,0.801397,0.600206


In [16]:
## P376
fig = px.imshow(corr_youtube.select(pl.all().round(2)), 
    y = corr_youtube.columns,
    text_auto=True, aspect="auto", color_continuous_scale='RdBu_r')
fig.show()

In [17]:
## P377
corr_by_country = con_youtube.sql("""
    SELECT country, corr(view_count, like_count) AS pearson_corr
    FROM youtube
    GROUP BY country
    ORDER BY pearson_corr DESC; """).pl()
corr_by_country

country,pearson_corr
str,f64
"""KR""",0.932545
"""JP""",0.926643
"""YE""",0.920381
"""ID""",0.919457
"""MA""",0.912936
…,…
"""MX""",0.521178
"""PG""",0.392778
"""US""",0.392224


In [ ]:
## P378
corr_by_3country = con_youtube.sql("""
    SELECT country, view_count, like_count
    FROM youtube
    WHERE country IN ('DE', 'US', 'AT') """).pl()
fig = px.scatter(corr_by_3country, x = 'view_count', y = 'like_count' , color = 'country', opacity=0.5)
fig.show()

In [19]:
## P379
con_youtube.sql("""
    SELECT title, snapshot_date, country, view_count, like_count
    FROM youtube
    WHERE view_count > 1_400_000_000 """).pl()

title,snapshot_date,country,view_count,like_count
str,date,str,i64,i64
"""Discord Loot Boxes are here.""",2024-04-03,"""HN""",1405652977,141970
"""Discord Loot Boxes are here.""",2024-04-04,"""HN""",1406473877,170478
"""Discord Loot Boxes are here.""",2024-04-03,"""PE""",1405662862,142469
"""Discord Loot Boxes are here.""",2024-04-04,"""PE""",1406478891,170682
"""Discord Loot Boxes are here.""",2024-04-02,"""US""",1407108038,69908
…,…,…,…,…
"""Discord Loot Boxes are here.""",2024-04-04,"""MX""",1406476681,170632
"""Discord Loot Boxes are here.""",2024-04-03,"""CO""",1405646950,141709
"""Discord Loot Boxes are here.""",2024-04-04,"""CO""",1406472799,170387


In [20]:
## P379 하단
con_youtube.sql("""
    SELECT country, corr(view_count, like_count) AS pearson_corr
    FROM youtube
    WHERE view_count < 1400000000
    GROUP BY country
    ORDER BY pearson_corr DESC; """).pl()

country,pearson_corr
str,f64
"""KR""",0.932545
"""JP""",0.926643
"""YE""",0.920381
"""ID""",0.919457
"""MA""",0.912936
…,…
"""SV""",0.838851
"""NP""",0.827126
"""PK""",0.825395


## 9.5 주제별 유튜브 트렌드 분석

In [21]:
## P381
con_youtube.sql("""
    FROM youtube
    SELECT count() AS "데이터 수", min(snapshot_date) AS 시작일,
        max(snapshot_date) AS 최종일, 최종일-시작일 AS "수집 기간"
    WHERE country = 'KR'; """).pl()

데이터 수,시작일,최종일,수집 기간
i64,date,date,i64
39796,2023-10-26,2025-12-31,797


In [22]:
### P382
con_youtube.sql("""
    WITH ranked_videos AS (
        FROM youtube
        SELECT *, yearweek: date_part('yearweek', "snapshot_date"),
        WHERE daily_rank = 1 AND country = 'KR')
    FROM (SELECT *,
        ROW_NUMBER() OVER (PARTITION BY yearweek ORDER BY view_count DESC) AS rn
        FROM ranked_videos)
    SELECT "연도-주": yearweek::string, 제목: title, 채널: channel_name, "재생 수": view_count,
        수집일: snapshot_date
    WHERE rn = 1
    ORDER BY view_count DESC
    LIMIT 10; """).pl()

연도-주,제목,채널,재생 수,수집일
str,str,str,i64,date
"""202429""","""50 YouTubers Fight For $1,000,…","""MrBeast""",120507144,2024-07-16
"""202432""","""Survive 100 Days In Nuclear Bu…","""MrBeast""",88837531,2024-08-05
"""202443""","""ROSÉ & Bruno Mars - APT. (Off…","""ROSÉ""",72745889,2024-10-21
"""202402""","""Still Here | Season 2024 Cinem…","""League of Legends""",69594735,2024-01-14
"""202507""","""I Spent 100 Hours Inside The P…","""MrBeast""",62944581,2025-02-10
"""202352""","""I Rescued 100 Abandoned Dogs!""","""MrBeast""",59622909,2023-12-27
"""202442""","""ROSÉ & Bruno Mars - APT. (Off…","""ROSÉ""",56125246,2024-10-20
"""202528""","""BLACKPINK - ‘뛰어(JUMP)’ M/V""","""BLACKPINK""",43056366,2025-07-13
"""202426""","""LISA - ROCKSTAR (Official Musi…","""LLOUD Official""",40420776,2024-06-29


In [23]:
## P383
con_youtube.sql("""
    WITH ranked_videos AS (
        FROM youtube
        SELECT *, yearweek: date_part('yearweek', "snapshot_date"),
        WHERE daily_rank = 1 AND country = 'KR')
    FROM (SELECT *,
        ROW_NUMBER() OVER (PARTITION BY yearweek ORDER BY view_count DESC) AS rn
        FROM ranked_videos)
    SELECT 연도주: yearweek::string, 제목: title, 채널: channel_name, "재생 수": view_count,
    수집일: snapshot_date
    WHERE rn = 1 AND REGEXP_MATCHES(title, '[가-힣]')
    ORDER BY view_count DESC
    LIMIT 10; """).pl()

연도주,제목,채널,재생 수,수집일
str,str,str,i64,date
"""202528""","""BLACKPINK - ‘뛰어(JUMP)’ M/V""","""BLACKPINK""",43056366,2025-07-13
"""202420""","""aespa 에스파 'Supernova' MV""","""SMTOWN""",29158119,2024-05-15
"""202435""","""LE SSERAFIM (르세라핌) 'CRAZY' OFF…","""HYBE LABELS""",26524326,2024-09-01
"""202536""","""aespa 에스파 'Rich Man' MV""","""SMTOWN""",23222996,2025-09-07
"""202418""","""IVE 아이브 '해야 (HEYA)' MV""","""STARSHIP""",21849530,2024-05-02
"""202526""","""aespa 에스파 'Dirty Work' MV""","""SMTOWN""",20166315,2025-06-29
"""202522""","""SEVENTEEN (세븐틴) 'THUNDER' Offi…","""HYBE LABELS""",18568601,2025-05-27
"""202544""","""LE SSERAFIM (르세라핌) 'SPAGHETTI …","""HYBE LABELS""",17257306,2025-10-27
"""202344""","""정국 (Jung Kook) 'Standing Next …","""HYBE LABELS""",15696096,2023-11-05


In [24]:
## P384
con_youtube.sql("""
    WITH korean_videos AS (
        FROM youtube
        SELECT RANK: RANK() OVER (PARTITION BY country ORDER BY like_count DESC), *
        WHERE daily_rank = 1 AND country != 'KR' AND REGEXP_MATCHES(title, '[가-힣]'))
    FROM (FROM korean_videos
        SELECT video_id, channel_id, country, snapshot_date: max(snapshot_date)
        GROUP BY ALL) AS KOREAN
    LEFT JOIN korean_videos USING (video_id, channel_id, country, snapshot_date)
    SELECT country, title, channel_name, snapshot_date, RANK, view_count, like_count
    ORDER BY view_count DESC """).pl()

country,title,channel_name,snapshot_date,RANK,view_count,like_count
str,str,str,date,i64,i64,i64
"""MY""","""BLACKPINK - ‘뛰어(JUMP)’ M/V""","""BLACKPINK""",2025-08-02,1,120454066,5160907
"""SG""","""BLACKPINK - ‘뛰어(JUMP)’ M/V""","""BLACKPINK""",2025-08-01,1,117882675,5135885
"""MN""","""BLACKPINK - ‘뛰어(JUMP)’ M/V""","""BLACKPINK""",2025-07-31,1,115294090,5109968
"""IS""","""BLACKPINK - ‘뛰어(JUMP)’ M/V""","""BLACKPINK""",2025-07-29,1,110056862,5056518
"""BE""","""BLACKPINK - ‘뛰어(JUMP)’ M/V""","""BLACKPINK""",2025-07-26,1,101933108,4963304
…,…,…,…,…,…,…
"""ID""","""XngHan&Xoul 승한앤소울 'Waste No Ti…","""SMTOWN""",2025-08-01,48,363928,73561
"""GB""","""CORTIS (코르티스) 'FaSHioN' Offici…","""HYBE LABELS""",2025-09-08,15,301529,36984
"""ID""","""VVUP (비비업) ‘House Party’ MV""","""VVUP""",2025-10-23,52,249571,49119


In [25]:
## P385
con_youtube.sql("""
    FROM (FROM youtube
        SELECT country, video_id, channel_id
        WHERE daily_rank = 1 AND country != 'KR' AND REGEXP_MATCHES(title, '[가-힣]')
    GROUP BY country, video_id, channel_id)
    SELECT country, count()
    GROUP BY country
    ORDER BY #2 DESC
    LIMIT 10; """).pl().transpose(include_header=True)

column,column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9
str,str,str,str,str,str,str,str,str,str,str
"""country""","""ID""","""JP""","""TW""","""CA""","""BO""","""GB""","""GT""","""VE""","""TH""","""RU"""
"""count_star()""","""42""","""30""","""21""","""16""","""15""","""13""","""12""","""11""","""11""","""11"""


In [26]:
## P386
con_youtube.sql("""FROM youtube
    SELECT channel_name AS "채널 이름", title AS "영상 제목",
        count() AS "순위 일수", min(snapshot_date) AS "순위 시작일",
        max(snapshot_date) AS "순위 종료일", min(daily_rank) AS "최고 순위",
        max(view_count) AS "최대 재생 수", max(like_count) AS "최대 좋아요 수",
        max(comment_count) AS "최대 댓글 수"
    WHERE country = 'KR'
    GROUP BY channel_id, "채널 이름", video_id, "영상 제목"
    ORDER BY ("순위 일수", "최대 재생 수") DESC
    LIMIT 10; """).pl()

채널 이름,영상 제목,순위 일수,순위 시작일,순위 종료일,최고 순위,최대 재생 수,최대 좋아요 수,최대 댓글 수
str,str,i64,date,date,i64,i64,i64,i64
"""유우키의 일본이야기 YUUKI""","""24시간동안 도쿄 슬럼가 음식으로 끼니 해결해보기""",12,2024-12-25,2025-01-05,5,1023982,19045,1662
"""MrBeast""","""Beat Ronaldo, Win $1,000,000""",11,2024-12-01,2024-12-11,4,141409497,6301933,95394
"""SBS Entertainment""","""2NE1(투애니원) - Come Back Home+Fi…",11,2024-12-26,2025-01-05,1,5932946,236127,16196
"""SBS Entertainment""","""G-DRAGON(지드래곤) - POWER+맨정신+삐딱하…",11,2024-12-26,2025-01-05,1,4292869,132664,13700
"""숏박스""","""상견례""",11,2025-04-26,2025-05-06,1,3972743,57531,3598
"""코마""","""시청자 100명 속에 잠입하기 (재도전)""",11,2025-04-26,2025-05-06,2,1475174,26995,3173
"""빠니보틀 Pani Bottle""","""꼬마 가이드와 함께한 마닐라 쓰레기산 슬럼가 투어 【필…",11,2024-12-17,2024-12-27,4,1287248,21839,1951
"""찰스엔터""","""퀸가비 선배 집에서 노가리(with 승헌쓰)""",11,2024-12-25,2025-01-04,4,1051757,29533,1950
"""찰스엔터""","""[vlog] 승헌,준빵,소정과 플러…",11,2024-11-28,2024-12-08,4,778649,20715,1787


In [27]:
## P387
con_youtube.sql("""
    FROM (FROM youtube
        SELECT country, channel_id, video_id, "순위 일수": count()
        GROUP BY country, channel_id, video_id)
    SELECT country, "평균 순위 일수": avg("순위 일수").round(5),
    순위: rank() over (order by "평균 순위 일수")
    GROUP BY country
    ORDER BY "평균 순위 일수"
    LIMIT 10 """).pl().transpose(include_header=True)

column,column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9
str,str,str,str,str,str,str,str,str,str,str
"""country""","""RU""","""PG""","""US""","""GB""","""CA""","""IN""","""BR""","""DE""","""TH""","""FR"""
"""평균 순위 일수""","""1.39487""","""1.69797""","""1.69938""","""1.71729""","""1.76457""","""1.88139""","""1.88377""","""1.90601""","""1.94978""","""2.09787"""
"""순위""","""1""","""2""","""3""","""4""","""5""","""6""","""7""","""8""","""9""","""10"""


In [28]:
## P387하단
days_in_ranking = con_youtube.sql("""
    FROM youtube
    SELECT country, channel_id, video_id, \"순위 일수\": count()
    WHERE country in ('KR', 'RU', 'US')
    GROUP BY country, channel_id, video_id""").pl()
fig = px.box(days_in_ranking, x = 'country', y = '순위 일수')
fig.show()

In [29]:
## P389
con_youtube.sql("""
    FROM youtube
    JOIN (FROM youtube
        SELECT channel_id, video_id, snapshot_date: min(snapshot_date),
        duration: min(snapshot_date) - first(publish_date)::DATE
        GROUP BY channel_id, video_id) USING (channel_id, video_id, snapshot_date)
    SELECT channel_name, title, daily_rank, snapshot_date, duration
    WHERE country = 'KR'
    ORDER BY duration, daily_rank """).pl()

channel_name,title,daily_rank,snapshot_date,duration
str,str,i64,date,i64
"""JennieRubyJaneVEVO""","""JENNIE - like JENNIE (Official…",1,2025-03-07,0
"""HYBE LABELS""","""LE SSERAFIM (르세라핌) 'SPAGHETTI …",1,2025-10-24,0
"""임영웅""","""임영웅 '알겠어요 미안해요' Official M/V""",1,2025-12-08,0
"""BLACKPINK""","""BLACKPINK - ‘뛰어(JUMP)’ M/V""",1,2025-07-11,0
"""스포타임""","""이강인, PSG·UCL 데뷔골 터졌다!… PSG는 3대…",1,2023-10-26,0
…,…,…,…,…
"""My Boy Arlo - Topic""","""Midnight Alchemy""",47,2025-11-24,27
"""1MILLION - Topic""","""Can We Luv""",21,2025-08-09,30
"""Republic Records""","""“Takedown” KPop Demon Hunters …",32,2025-07-26,30


In [30]:
## P389
con_youtube.sql("""
    WITH first_rank AS (
        FROM youtube
        JOIN (FROM youtube
            SELECT channel_id, video_id, snapshot_date: min(snapshot_date),
                duration: min(snapshot_date) - first(publish_date)::DATE
            GROUP BY channel_id, video_id) USING (channel_id, video_id, snapshot_date)
        SELECT * EXCLUDE (daily_movement, weekly_movement, country, description, thumbnail_url, 
video_id, channel_id, video_tags)
        WHERE country = 'KR')
    FROM first_rank
    SELECT daily_rank, "순위별 영상 수": count(), "평균 재생 수": avg(view_count).round(1),
        "최대 재생 수": max(view_count), "최소 재생 수": min(view_count),
        "표준편차(백만)": stddev(view_count)/1_000_000 GROUP BY daily_rank
    LIMIT 10; """).pl()

daily_rank,순위별 영상 수,평균 재생 수,최대 재생 수,최소 재생 수,표준편차(백만)
i64,i64,f64,i64,i64,f64
1,452,1693924.9,23919817,26105,2.206022
2,441,1145926.4,27056465,5187,1.936182
3,501,1152325.5,47355387,12257,2.544013
4,424,1034610.5,57745030,20416,3.634022
5,432,710799.9,12716665,3258,1.021924
6,477,661422.3,26560824,17586,1.817797
7,419,555379.4,6895650,20818,0.71273
8,407,511813.3,8095357,6710,0.687622
9,434,494090.4,14838699,9040,0.919752


In [31]:
## P391
first_rank = con_youtube.sql("""
    WITH first_rank AS (FROM youtube
        JOIN (FROM youtube
            SELECT channel_id, video_id, snapshot_date: min(snapshot_date),
                duration: min(snapshot_date) - first(publish_date)::DATE
            GROUP BY channel_id, video_id) USING (channel_id, video_id, snapshot_date)
            SELECT * EXCLUDE (daily_movement, weekly_movement, country, description, thumbnail_url, 
video_id, channel_id, video_tags)
            WHERE country = 'KR')
        FROM first_rank
        SELECT daily_rank, "순위별 영상 수": count(),
            "평균 재생 수": avg(view_count).round(1)
        GROUP BY daily_rank; """).pl()
fig = px.bar(first_rank, x = 'daily_rank', y = "평균 재생 수")
fig.show()

In [32]:
## P392
duration_view = con_youtube.sql("""
    FROM youtube
    JOIN (
        FROM youtube
        SELECT channel_id, video_id, snapshot_date: min(snapshot_date),
        duration: min(snapshot_date) - first(publish_date)::DATE
        GROUP BY channel_id, video_id) 
    USING (channel_id, video_id, snapshot_date)
    SELECT * EXCLUDE (daily_movement, weekly_movement, country, description, 
        thumbnail_url, video_id, channel_id, video_tags)
    WHERE country = 'KR' and duration <= 10""").aggregate(
    'duration, "평균 재생 수": avg(view_count).round(0), count: count()').pl()
fig = px.bar(duration_view, x = 'duration', y = '평균 재생 수', 
    custom_data=['count'], text='평균 재생 수')
fig.update_xaxes(title = '게시 후 경과일')
fig.update_traces(textfont_size = 11, texttemplate = '%{text:,}뷰<br>%{customdata[0]:,}개')
fig.show()

In [33]:
## P393
con_youtube.sql("""
    FROM youtube
    SELECT month: date_part('month', "publish_date"),
        like_rate: avg(like_rate).round(2)
    WHERE country = 'KR'
    GROUP BY month
    ORDER BY month """).pl().transpose(include_header=True)

column,column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""month""",1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0
"""like_rate""",2.78,2.88,3.22,3.2,3.24,3.06,3.02,3.02,2.91,3.19,3.0,3.05


In [34]:
## P394
from scipy.stats import f_oneway
ANOVA_raw = con_youtube.sql("""
    SELECT month: date_part('month', "publish_date"), like_rate
    FROM youtube
    WHERE country = 'KR'""").pl()
ANOVA_groups = [ ANOVA_groups['like_rate'].to_numpy()
    for key, ANOVA_groups in ANOVA_raw.partition_by('month', as_dict=True).items()]
f_stat, p_value = f_oneway(*ANOVA_groups)
print(f"F-statistic: {f_stat:.3f}, p-value: {p_value:.4f}")

F-statistic: 6.559, p-value: 0.0000


## 9.6 <케이팝 데몬 헌터스> 유튜브 트랜드 분석

In [35]:
## P407
import duckdb
import polars as pl
import plotly.express as px
con_KDH = duckdb.connect("./youtube.duckdb")

In [36]:
## P407~408
import pycountry_convert as pc
def get_continent_name(nation_code: str) -> str:
   continent_code = pc.country_alpha2_to_continent_code(nation_code)
   continent_dict = {"NA": "North America", "SA": "South America", "AS": "Asia", "AF": "Africa",
                     "OC": "Oceania", "EU": "Europe", "AQ": "Antarctica"}
   return continent_dict[continent_code]
con_KDH.create_function("country_alpha2_to_continent_code", get_continent_name)

import pycountry
def get_country_name(alpha2: str) -> str:
   country = pycountry.countries.get(alpha_2=alpha2.upper())
   return country.name if country else None
con_KDH.create_function("alpha2_to_country_name", get_country_name)

def get_alpha3(code2: str) -> str:
   country = pycountry.countries.get(alpha_2=code2.upper())
   return country.alpha_3
con_KDH.create_function("alpha2_to_alpha3", get_alpha3)

In [ ]:
## P408하단
con_KDH.execute("""
    ALTER TABLE k_de_hun ADD COLUMN continent varchar;
    ALTER TABLE k_de_hun ADD COLUMN country_name varchar;
    ALTER TABLE k_de_hun ADD COLUMN alpha3 varchar;
    UPDATE k_de_hun SET continent = country_alpha2_to_continent_code(country);
    UPDATE k_de_hun SET country_name = alpha2_to_country_name(country);
    UPDATE k_de_hun SET alpha3 = alpha2_to_alpha3(country); """)

In [38]:
## P409
con_KDH.sql("""FROM k_de_hun
    SELECT 대륙:continent, 국가명:list(DISTINCT country_name), 국가수:length(국가명),
    GROUP BY 대륙;""").pl()

대륙,국가명,국가수
str,list[str],i64
"""Oceania""","[""Australia"", ""New Zealand"", ""Papua New Guinea""]",3
"""North America""","[""Mexico"", ""Costa Rica"", … ""Honduras""]",12
"""South America""","[""Paraguay"", ""Uruguay"", … ""Brazil""]",10
"""Asia""","[""Hong Kong"", ""Cambodia"", … ""Indonesia""]",21
"""Africa""","[""South Africa"", ""Nigeria"", … ""Ghana""]",7
"""Europe""","[""Russian Federation"", ""Austria"", … ""Czechia""]",19


In [39]:
## P409하단
con_KDH.sql("""FROM k_de_hun
    SELECT 대륙:continent, 국가명:list(DISTINCT country_name), 국가수:length(국가명),
    WHERE video_id = 'yebNIHKAC4A'
    GROUP BY 대륙;""").pl()

대륙,국가명,국가수
str,list[str],i64
"""Europe""","[""Austria"", ""United Kingdom"", … ""France""]",4
"""Oceania""","[""Papua New Guinea"", ""Australia""]",2
"""Asia""","[""Malaysia"", ""Indonesia"", … ""Korea, Republic of""]",5
"""North America""","[""Mexico"", ""Costa Rica"", … ""Guatemala""]",9
"""South America""","[""Paraguay"", ""Venezuela, Bolivarian Republic of"", … ""Bolivia, Plurinational State of""]",9


In [40]:
## P410
df = con_KDH.sql("""
    FROM (FROM k_de_hun
        SELECT video_id, duration:max(snapshot_date)::DATE - min(publish_date)::DATE + 1,
        평균조회:round(max(view_count)/(duration), 0)
        WHERE country = 'KR'
        GROUP BY video_id
        ORDER BY #3 desc
        LIMIT 10) JOIN k_de_hun USING (video_id)
        SELECT video_id, title, snapshot_date, daily_rank, view_count, like_count, publish_date, 
duration, 평균조회
        WHERE country = 'KR' """).pl()

fig = px.line(df, x = 'like_count', y = 'view_count', color ='title', line_group='video_id')
fig.update_xaxes(title='좋아요 수')
fig.update_yaxes(title='조회 수')
fig.update_layout(
    legend=dict(
        orientation='h', # 수평(horizontal) 배치
        yanchor='top', # 범례 박스의 위쪽 기준으로
        y=-0.25, # 그래프 아래쪽에 배치 (값을 조정하며 위치 조절)
        xanchor='center', # 중앙 기준
        x=0.5 )) # 가운데 정렬

fig.show()

In [41]:
## P411
import plotly.graph_objects as go
## 제목이 너무 길어 일부 문자열 삭제를 위한 정규 표현식 정의
pattern = r"(?i)(Kpop Demon Hunters|Sony Animation|케이팝 데몬 헌터스|Netflix Family|넷플릭스|\|)"
df = (con_KDH.sql("""
    FROM (FROM k_de_hun
        SELECT video_id, duration:max(snapshot_date)::DATE - min(publish_date)::DATE + 1,
            평균조회:round(max(view_count)/(duration), 0)
        WHERE country = 'KR'
        GROUP BY video_id, title
        ORDER BY 3 DESC
       LIMIT 10) JOIN k_de_hun USING (video_id)
    SELECT first(title) AS 제목, min(snapshot_date) AS "순위 진입일",
        max(snapshot_date) AS "순위 최종일",
        first(view_count) AS "최초 조회 수", last(view_count) AS "최종 조회 수"
    WHERE country = 'KR'
    GROUP BY video_id
    ORDER BY "최종 조회 수";""").pl().with_columns(pl.col('제목').str.replace_all(pattern, 
"")))
titles, x0, x1, v0, v1 = [df[c].to_list() for c in
    ["제목","순위 진입일","순위 최종일","최초 조회 수","최종 조회 수"]]
gmax = max(max(v0), max(v1)) or 1
scale = lambda xs: [6 + 22*(x/gmax) for x in xs] ## 6~28px
s0, s1 = scale(v0), scale(v1)

fig = go.Figure([
    *[go.Scatter(x=[a,b], y=[t,t], mode="lines",
        line=dict(color="rgba(60,60,60,0.6)", width=4), showlegend=False, 
        hoverinfo="skip")
    for t,a,b in zip(titles, x0, x1)],
        go.Scatter(x=x0, y=titles, mode="markers", name="첫날 조회 수",
            marker=dict(size=s0, line=dict(width=1,color="white"), color="#f4c542")),
        go.Scatter(x=x1, y=titles, mode="markers", name="마지막날 조회 수",
            marker=dict(size=s1, line=dict(width=1,color="white"), color="#6c83ff")) ])

fig.update_layout(
    title="순위 기간의 조회 수 변화",
    xaxis_title="날짜",
    yaxis=dict(type="category", automargin=True, categoryorder="array", categoryarray=titles),
    template="plotly_white", legend_title_text="")

fig.show()

In [42]:
## P413
df = con_KDH.sql("""
    FROM k_de_hun
    SELECT alpha3, country_name, continent, 영상수:count()
    GROUP BY ALL""").pl()

fig = px.scatter_geo(
    df, locations='alpha3',
    size='영상수', color='continent',
    hover_name='country_name',
    title=" 케이팝 데몬 헌터스 관련 영상 개수")
fig.update_layout(legend_title_text='')
fig.show()

In [43]:
## P414
summary = con_KDH.sql("""
    FROM k_de_hun
    SELECT video_id, title, country_name, continent,
        "최종 조회 수":MAX(view_count), "최종 좋아요 수":MAX(like_count),
        "평균 좋아요 비율":ROUND(100.0 * "최종 좋아요 수" / "최종 조회 수",2)
    GROUP BY ALL""").pl()

fig_views = px.treemap(summary, path=['continent', 'country_name', 'title'],
    values='최종 좋아요 수', color='평균 좋아요 비율',
    color_continuous_scale='Blues', title="케이팝 데몬 헌터스 관련 영상 좋아요 수 트리맵")

fig_views.update_layout(margin=dict(l=10, r=10, t=60, b=10))
fig_views.show()